# LocalFood AI — Agentic AI demonstrations (T13)

This notebook runs the **same Python tools and explicit plan → act → observe → decide loop** used by the web app.
Every cell prints the evidence required for a viva:
- The **two required tools**: `find_restaurants(city)` and `filter_by_cuisine(type)`
- **Session memory** that carries dietary preferences across turns
- **Dietary guardrail** that refuses unsafe recommendations
- **Multi-step trace** showing each decision the agent made

Run all cells top-to-bottom (`Kernel → Restart & Run All`).

In [ ]:
import json
import os
import sys
from pprint import pprint

sys.path.insert(0, os.path.abspath("backend"))
from agent import run_agent
from memory import clear_memory, get_memory

def banner(title):
    width = 60
    print(f"\n{'─' * width}")
    print(f"  {title}")
    print(f"{'─' * width}")

def show(label, value):
    print(f"\n{'=' * 12} {label} {'=' * 12}")
    if isinstance(value, (dict, list)):
        pprint(value, sort_dicts=False)
    else:
        print(value)

def show_trace(result):
    """Print a human-readable PLAN → ACT → OBSERVE → DECIDE trace."""
    print()
    print("AGENT TRACE (multi-step)")
    print("-" * 50)
    for step in result["trace"]:
        tag = f"[{step['type'].upper():<10}]"
        print(f"  STEP {step['step']:>2} {tag} {step['title']}")
        print(f"          {step['detail'][:90]}")
        if step.get("tool"):
            print(f"          → TOOL CALL : {step['tool']}({step.get('arguments', {})}")
            print(f"          → TOOL RESULT: {step.get('result', '')}")
    print("-" * 50)
    print(f"FINAL REPLY: {result['reply']}")
    stats = result.get("stats", {})
    print(f"STATS      : searched={stats.get('searched')}  "
          f"after_cuisine={stats.get('afterCuisine')}  "
          f"after_diet={stats.get('afterDiet')}  "
          f"elapsed={stats.get('elapsedMs')}ms")

print("LocalFood AI notebook ready — T13 demo")

---
## Demo 1 — Full recommendation (both tools fire)

**Input:** `"I'm vegetarian and I want spicy Punjabi food in Jalandhar."`

**Expected multi-step trace:**
1. Extract intent + preferences
2. Read memory (empty — fresh session)
3. Write memory (diet=vegetarian, cuisine=Punjabi, location=Jalandhar)
4. Form plan
5. **TOOL CALL** → `find_restaurants("Jalandhar")` → observe result
6. **TOOL CALL** → `filter_by_cuisine("Punjabi")` → observe result
7. Apply dietary guardrail (keep vegetarian-compatible only)
8. Rank and return top-5

In [ ]:
banner("DEMO 1 — Full agentic recommendation (both tools)")
session = "notebook-demo-1"
clear_memory(session)

user_input = "I'm vegetarian and I want spicy Punjabi food in Jalandhar."
show("USER INPUT", user_input)

result = run_agent(session, user_input)

show("MEMORY WRITTEN", result["memory"])
show_trace(result)

# Confirm both tools fired
tools_used = [step["tool"] for step in result["trace"] if step.get("tool")]
print(f"\nTools that fired: {tools_used}")
assert "find_restaurants" in tools_used, "find_restaurants must have been called"
assert "filter_by_cuisine" in tools_used, "filter_by_cuisine must have been called"
print("✓ Both required tools executed")

if result["recommendations"]:
    best = result["recommendations"][0]
    show("TOP RECOMMENDATION", best)
    print(f"\n✓ Top pick: {best['name']}  |  Score: {best['score']}/100  |  Diet: {best['diet']}")
else:
    print("No recommendations returned.")

---
## Demo 2 — Memory carried across turns

**Turn 1:** `"I'm vegetarian and I like Punjabi food."`  (no city — agent asks for it)

**Turn 2:** `"Find something in Jalandhar."`  (no diet, no cuisine repeated)

The agent must read Turn 1's memory and apply `diet=vegetarian` + `preferredCuisine=Punjabi`
as hard constraints on Turn 2's search — without the user repeating themselves.

In [ ]:
banner("DEMO 2 — Cross-turn memory (vegetarian + Punjabi → Jalandhar)")
session = "notebook-demo-2"
clear_memory(session)

# ── Turn 1: preferences only, no city ───────────────────────────────────
print("TURN 1")
turn1 = run_agent(session, "I'm vegetarian and I like Punjabi food.")
show("TURN 1 REPLY", turn1["reply"])
show("MEMORY AFTER TURN 1", get_memory(session))
assert get_memory(session)["diet"] == "vegetarian", "diet must be remembered"
assert get_memory(session)["preferredCuisine"] == "Punjabi", "cuisine must be remembered"
print("✓ Turn 1: vegetarian + Punjabi stored in memory")

# ── Turn 2: city only — memory provides diet + cuisine ───────────────────
print("\nTURN 2")
turn2_input = "Find something in Jalandhar."
show("TURN 2 USER INPUT", turn2_input)
turn2 = run_agent(session, turn2_input)

show_trace(turn2)

# Verify memory was applied
print(f"\nMemory used for Turn 2:")
print(f"  diet             = {turn2['memory']['diet']}")
print(f"  preferredCuisine = {turn2['memory']['preferredCuisine']}")
print(f"  location         = {turn2['memory']['location']}")

assert turn2["memory"]["diet"] == "vegetarian", "diet from Turn 1 must carry to Turn 2"
assert turn2["memory"]["preferredCuisine"] == "Punjabi", "cuisine from Turn 1 must carry to Turn 2"
print("✓ Turn 2 used remembered vegetarian + Punjabi without them being re-stated")

# Confirm tools fired
tools_used = [step["tool"] for step in turn2["trace"] if step.get("tool")]
assert "find_restaurants" in tools_used
assert "filter_by_cuisine" in tools_used
print("✓ Both tools executed on Turn 2")

---
## Demo 3 — Dietary conflict (honest refusal)

The agent remembers `diet=vegetarian` from Turn 1.  
Turn 2 asks for a non-vegetarian restaurant.  
The agent **must refuse before calling any tools** — hard constraints take priority.

This demonstrates the dietary guardrail and honest failure handling.

In [ ]:
banner("DEMO 3 — Dietary conflict (safe refusal, no fabricated result)")
session = "notebook-demo-3"
clear_memory(session)

run_agent(session, "I'm vegetarian.")

conflict_input = "I want a non-vegetarian restaurant in Jalandhar."
show("TURN 2 INPUT", conflict_input)
show("MEMORY AT TIME OF CONFLICT", get_memory(session))

conflict = run_agent(session, conflict_input)
show_trace(conflict)

print(f"\nRecommendations returned: {len(conflict['recommendations'])}")
assert len(conflict["recommendations"]) == 0, "must return 0 recommendations on conflict"

# Verify no tool was called
tools_called = [step["tool"] for step in conflict["trace"] if step.get("tool")]
print(f"Tools called: {tools_called}")
assert not tools_called, "no tools should fire when a dietary conflict is detected"
print("✓ Agent refused safely — 0 recommendations, no tools called")

---
## Demo 4 — No-match (honest failure)

User asks for Jain-friendly Chinese food in Jalandhar.  
The dataset has no entry that satisfies both constraints.  
The agent must report the honest failure — not fabricate a result.

This covers T13's requirement for demonstrating a failure case.

In [ ]:
banner("DEMO 4 — Honest no-match (Jain + Chinese in Jalandhar)")
session = "notebook-demo-4"
clear_memory(session)

user_input = "I follow Jain dietary rules. Find me Chinese food in Jalandhar."
show("USER INPUT", user_input)

result = run_agent(session, user_input)
show_trace(result)

print(f"\nRecommendations returned: {len(result['recommendations'])}")

# Both tools should still fire — the failure happens AFTER the search
tools_used = [step["tool"] for step in result["trace"] if step.get("tool")]
print(f"Tools that fired: {tools_used}")
assert "find_restaurants" in tools_used, "find_restaurants must still be called"

print("✓ Agent honestly reported no matching result — no fabrication")
print(f"   Agent reply: {result['reply']}")